# 🏪 Alura Store Latam — Análisis de Rendimiento

**Objetivo:** Analizar el rendimiento de 4 tiendas mediante métricas de facturación, categorías de venta, calificaciones y tendencias temporales, para identificar cuál tienda tiene el menor desempeño y recomendar una acción estratégica.

---

## 📦 0. Importación de librerías

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 📥 1. Importación de datos

Cargamos los datos de las 4 tiendas desde el repositorio de GitHub de Alura.

In [ ]:
url  = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_1%20.csv"
url2 = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_2.csv"
url3 = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_3.csv"
url4 = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_4.csv"

df1 = pd.read_csv(url)
df2 = pd.read_csv(url2)
df3 = pd.read_csv(url3)
df4 = pd.read_csv(url4)

print("Datos cargados correctamente.")
df1.head()

## 🔧 2. Preparación del DataFrame consolidado

Agregamos una columna `Tienda_ID` a cada DataFrame y los concatenamos en uno solo (`df_total`) para facilitar los análisis comparativos.

In [ ]:
df1['Tienda_ID'] = 'Tienda 1'
df2['Tienda_ID'] = 'Tienda 2'
df3['Tienda_ID'] = 'Tienda 3'
df4['Tienda_ID'] = 'Tienda 4'

df_total = pd.concat([df1, df2, df3, df4], ignore_index=True)

# Asegurar tipos correctos
df_total['Costo de envío']  = df_total['Costo de envío'].astype(float)
df_total['Fecha de Compra'] = pd.to_datetime(df_total['Fecha de Compra'], dayfirst=True)
df_total['Lucro_Hipotetico'] = df_total['Precio'] - df_total['Costo de envío']

print(f"Registros totales: {len(df_total):,}")
df_total.info()

## 💰 3. Análisis de facturación

Calculamos los ingresos totales (suma de precios de venta) por tienda y el acumulado global.

In [ ]:
for i, df in enumerate([df1, df2, df3, df4], 1):
    print(f"Facturación Tienda {i}: ${df['Precio'].astype(float).sum():,.2f}")

facturacion_total = sum(df['Precio'].astype(float).sum() for df in [df1, df2, df3, df4])
print(f"\nFacturación total de todas las tiendas: ${facturacion_total:,.2f}")

## 🗂️ 4. Ventas por categoría de producto

Desglosamos los ingresos por categoría para cada tienda.

In [ ]:
def ventas_por_categoria(df, numero):
    ventas = df.groupby('Categoría del Producto')['Precio'].sum()
    print(f"\nVentas por categoría en Tienda {numero}:")
    print(ventas)

for i, df in enumerate([df1, df2, df3, df4], 1):
    ventas_por_categoria(df, i)

## ⭐ 5. Calificación promedio por tienda

Analizamos la satisfacción del cliente a través del promedio de calificaciones.

In [ ]:
def clasificacion_promedio(df, numero):
    promedio = df['Calificación'].mean()
    print(f"Clasificación promedio en Tienda {numero}: {promedio:.2f}")

for i, df in enumerate([df1, df2, df3, df4], 1):
    clasificacion_promedio(df, i)

## 🏆 6. Productos más y menos vendidos

Identificamos los productos con mayor y menor número de ventas en cada tienda.

In [ ]:
def productos_mas_menos(df, numero):
    conteo = df['Producto'].value_counts()
    print(f"\nTienda {numero}")
    print(f"  ✅ Más vendido:   {conteo.idxmax()} ({conteo.max()} unidades)")
    print(f"  ❌ Menos vendido: {conteo.idxmin()} ({conteo.min()} unidades)")

for i, df in enumerate([df1, df2, df3, df4], 1):
    productos_mas_menos(df, i)

## 🚚 7. Costo de envío promedio por tienda

In [ ]:
envio_promedio = df_total.groupby('Tienda_ID')['Costo de envío'].mean()
print("\nEnvío promedio por tienda:")
for t, p in envio_promedio.items():
    print(f"  {t}: ${p:,.2f}")

---
## 📊 8. Gráficos de análisis

Paleta de colores: **rojo** para la Tienda 4 (menor rendimiento), azul/naranja/verde para las demás.

In [ ]:
colores   = ['#1f77b4', '#ff7f0e', '#2ca02c', 'red']
color_map = {'Tienda 1': '#1f77b4', 'Tienda 2': '#ff7f0e',
             'Tienda 3': '#2ca02c',  'Tienda 4': 'red'}

### 📊 Gráfico 1 — Ingresos totales por tienda

In [ ]:
ingresos_totales = df_total.groupby('Tienda_ID')['Precio'].sum()

plt.figure(figsize=(8, 5))
ingresos_totales.plot(kind='bar', color=colores)
plt.title('Ingresos Totales por Tienda (Tienda 4 = Menor Ingreso)', fontsize=13)
plt.ylabel('Total Ingresos (USD)')
plt.xlabel('Tienda')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 📊 Gráfico 2 — Ingreso promedio por venta

In [ ]:
ingreso_promedio = df_total.groupby('Tienda_ID')['Precio'].mean()

plt.figure(figsize=(8, 5))
ingreso_promedio.plot(kind='bar', color=colores)
plt.title('Ingreso Promedio por Venta por Tienda')
plt.ylabel('Ingreso Promedio (USD)')
plt.xlabel('Tienda')
plt.grid(True)
plt.tight_layout()
plt.show()

### 📊 Gráfico 3 — Costo de envío vs. calificación del cliente

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_total, x='Calificación', y='Costo de envío', palette='Set2')
plt.title('Costo de Envío vs. Calificación del Cliente (Todas las Tiendas)')
plt.xlabel('Calificación del Cliente (1 a 5)')
plt.ylabel('Costo de Envío (USD)')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

### 📊 Gráfico 4 — Lucro hipotético mensual por tienda

In [ ]:
lucro_mensual = df_total.groupby(
    [df_total['Fecha de Compra'].dt.to_period('M'), 'Tienda_ID']
)['Lucro_Hipotetico'].sum().reset_index()
lucro_mensual['Fecha de Compra'] = lucro_mensual['Fecha de Compra'].astype(str)

plt.figure(figsize=(12, 6))
sns.lineplot(data=lucro_mensual, x='Fecha de Compra', y='Lucro_Hipotetico',
             hue='Tienda_ID', palette=color_map)
plt.title('Tendencia de Lucro Hipotético Mensual por Tienda', fontsize=15)
plt.ylabel('Lucro Hipotético Total Mensual (USD)')
plt.xlabel('Mes')
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

### 📊 Gráfico 5 — Tendencia suavizada de ventas (Media móvil 6 meses)

In [ ]:
ventas_mensuales = df_total.groupby(
    [df_total['Fecha de Compra'].dt.to_period('M'), 'Tienda_ID']
)['Precio'].sum().reset_index()
ventas_mensuales.columns = ['Fecha', 'Tienda', 'Ingresos_Mensuales']
ventas_mensuales['Fecha'] = ventas_mensuales['Fecha'].dt.to_timestamp()
ventas_mensuales = ventas_mensuales.set_index('Fecha')

plt.figure(figsize=(12, 6))
for t in ventas_mensuales['Tienda'].unique():
    data_t = ventas_mensuales[ventas_mensuales['Tienda'] == t].copy()
    data_t['Media_Movil_6M'] = data_t['Ingresos_Mensuales'].rolling(6).mean()
    sns.lineplot(data=data_t, x=data_t.index, y='Media_Movil_6M',
                 label=t, color=color_map[t])

plt.title('Tendencia Suavizada de Ventas Mensuales (Media Móvil 6M) — Destacando Tienda 4')
plt.ylabel('Ingreso Promedio Suavizado (USD)')
plt.xlabel('Fecha')
plt.grid(True)
plt.xticks(rotation=45)
plt.legend(title='Tienda')
plt.tight_layout()
plt.show()

### 📊 Gráfico 6 — Ingreso acumulado total por tienda

In [ ]:
plt.figure(figsize=(12, 6))
for t in ventas_mensuales['Tienda'].unique():
    data_t = ventas_mensuales[ventas_mensuales['Tienda'] == t].copy()
    data_t['Ingreso_Acumulado'] = data_t['Ingresos_Mensuales'].cumsum()
    sns.lineplot(data=data_t, x=data_t.index, y='Ingreso_Acumulado',
                 label=t, color=color_map[t])

plt.title('Ingreso Acumulado Total por Tienda — Destacando Tienda 4')
plt.ylabel('Ingreso Total Acumulado (USD)')
plt.xlabel('Fecha')
plt.grid(True)
plt.xticks(rotation=45)
plt.legend(title='Tienda')
plt.tight_layout()
plt.show()

---
## 📝 9. Informe de rendimiento y recomendación

### Hallazgos clave

| Métrica | Tienda 1 | Tienda 2 | Tienda 3 | Tienda 4 |
|---|---|---|---|---|
| Facturación total | $1,150,880,400 | $1,116,343,500 | $1,098,019,600 | **$1,038,375,700** |
| Calificación promedio | 3.98 | 4.04 | 4.05 | 4.00 |
| Envío promedio | $26,018 | $25,216 | $24,805 | **$23,459** |

### Conclusión

**La Tienda 4 presenta el menor rendimiento histórico** por las siguientes razones:

1. **Menor facturación total** (~$112M menos que Tienda 1) durante el período analizado.
2. **Menor ingreso promedio por venta** (~$440,000 USD), lo que indica una combinación de productos menos lucrativa.
3. **Tendencia a la baja sostenida** desde finales de 2021, siendo la única tienda con deterioro activo y continuo.
4. **Menor ingreso acumulado** a lo largo de todo el historial, posicionándola consistentemente como la de menor aporte al total.

### Recomendación

> **Vender o reestructurar la Tienda 4.** Sus problemas no son coyunturales sino estructurales: la tendencia negativa persiste a pesar de que el costo de envío es el más bajo del grupo, lo que descarta que el precio logístico sea la causa. Se recomienda auditar su mix de productos y evaluar si una intervención comercial puede revertir la tendencia antes de tomar una decisión de cierre.